# DSBM Training — Sentinel-1 → Sentinel-2

Run this notebook on **Kaggle** with the `sentinel12-image-pairs-segregated-by-terrain` dataset attached.

**Setup steps before running:**
1. Go to https://www.kaggle.com/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain
2. Click **New Notebook**
3. In the notebook: go to **Settings** (right panel) → **Accelerator** → select **GPU T4 x2**
4. Paste this notebook's contents in and run each cell


In [ ]:
# Step 1: Check GPU
import torch
if torch.cuda.is_available():
    print('GPU ready:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — enable it in Settings > Accelerator')

In [ ]:
# Step 2: Clone the code (removes old copy first to get latest version)
import os, shutil
if os.path.exists('/kaggle/working/dsbm-pytorch'):
    shutil.rmtree('/kaggle/working/dsbm-pytorch')
    print('Removed old clone')
os.chdir('/kaggle/working')
!git clone https://github.com/smilingsmilee/dsbm-pytorch.git
os.chdir('/kaggle/working/dsbm-pytorch')
print('Files:', os.listdir('.'))

In [ ]:
# Step 3: Install dependencies
!pip install -q hydra-core accelerate POT matplotlib seaborn torchmetrics torchdiffeq pytorch-lightning h5py torch-fidelity

In [ ]:
# Step 4: Set up data folders — auto-discover s1/s2 directories in the Kaggle dataset
import os, glob

os.makedirs('data/custom_source', exist_ok=True)
os.makedirs('data/custom_target', exist_ok=True)

# Find directories named 's1' and 's2' anywhere in the Kaggle input
s1_dirs = sorted([d for d in glob.glob('/kaggle/input/**/s1', recursive=True) if os.path.isdir(d)])
s2_dirs = sorted([d for d in glob.glob('/kaggle/input/**/s2', recursive=True) if os.path.isdir(d)])

if not s1_dirs or not s2_dirs:
    raise FileNotFoundError('Could not find s1/s2 directories. Check dataset is attached.')

# Use the first terrain type found (e.g. agri)
s1_dir = s1_dirs[0]
s2_dir = s2_dirs[0]
print(f'S1 source: {s1_dir}  ({len(os.listdir(s1_dir))} images)')
print(f'S2 source: {s2_dir}  ({len(os.listdir(s2_dir))} images)')

# Symlink as class subfolder (ImageFolder expects: root/classname/img.png)
if not os.path.exists('data/custom_source/images'):
    os.symlink(s1_dir, 'data/custom_source/images')
if not os.path.exists('data/custom_target/images'):
    os.symlink(s2_dir, 'data/custom_target/images')

print(f'Source images linked: {len(os.listdir("data/custom_source/images"))}')
print(f'Target images linked: {len(os.listdir("data/custom_target/images"))}')

In [ ]:
# Step 5: Run training
!python main.py dataset=custom_transfer method=dbdsb gamma_min=0.034 gamma_max=0.034 LOGGER=CSV device=cuda

In [ ]:
# Step 6: Save results to Kaggle output (so you can download them)
import shutil
shutil.copytree('experiments', '/kaggle/working/experiments', dirs_exist_ok=True)
print('Results saved to Kaggle output — use the Output tab to download')